> **Role:** Step 1 - one-time corpus embedding generation
> **Runtime:** Kaggle, 2x T4 GPU
> **Produces:** `binbaz_clean.csv`, `qwen3_06b_title_answer_embeddings.pt`, `embedding_metadata.json` (copy these into `data/`)

# Bin-Baz Fatwas — Embedding Generation (Kaggle, 2x T4)

**Purpose:** one-time batch job. Cleans the raw dataset, builds the final `binbaz_clean.csv`,
and generates Qwen3-Embedding-0.6B embeddings for `title + answer` (truncated to 1024 tokens
for embedding/reranking consistency — full answer text is preserved untouched for display).

**Run this on Kaggle** (Settings -> Accelerator -> GPU T4 x2) so both GPUs split the encoding work.

**Outputs (in `/kaggle/working/`, downloadable from Kaggle's Output tab once the run finishes):**
- `binbaz_clean.csv` — cleaned dataset with `fatwa_id`, `source`, `title`, `question`, `answer`,
  `embed_token_count`, `truncated`
- `qwen3_06b_title_answer_embeddings.pt` — embedding tensor, shape `(N, 1024)`, row-aligned to
  `binbaz_clean.csv` by `fatwa_id`
- `embedding_metadata.json` — provenance record (model, dim, truncation length, generation date)

### Structure
| # | Section |
|---|---|
| 1 | Setup & Imports |
| 2 | Configuration & Paths |
| 3 | Download Raw Dataset |
| 4 | Clean & Preprocess |
| 5 | Build `embed_text` + Truncation Metadata |
| 6 | Multi-GPU Embedding Generation |
| 7 | Save & Verify Outputs |


## 1 · Setup & Imports

In [1]:
!pip install -q sentence-transformers charset-normalizer ftfy opendatasets

import os, re, gc, json, time, unicodedata
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
from charset_normalizer import from_path as detect_encoding
import ftfy

print(f'GPUs available: {torch.cuda.device_count()}')
for i in range(torch.cuda.device_count()):
    print(f'  {i}: {torch.cuda.get_device_name(i)}  ({torch.cuda.get_device_properties(i).total_memory/1024**3:.1f} GB)')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.7 MB/s eta 0:00:00
GPUs available: 2
  0: Tesla T4  (14.6 GB)
  1: Tesla T4  (14.6 GB)


## 2 · Configuration & Paths

In [2]:
KAGGLE_DATASET = 'https://www.kaggle.com/datasets/a5medashraf/bin-baz-fatwas-dataset'
SOURCE_LABEL = 'Ibn Baz'   # scholar attribution — set per-dataset when you add more sources later

WORK_DIR = '/kaggle/working'
RAW_CSV       = "/kaggle/input/datasets/a5medashraf/bin-baz-fatwas-dataset/Full_BinBaz_Data.csv"
CLEAN_CSV     = os.path.join(WORK_DIR, 'binbaz_clean.csv')
EMBEDDINGS_PT = os.path.join(WORK_DIR, 'qwen3_06b_title_answer_embeddings.pt')
METADATA_JSON = os.path.join(WORK_DIR, 'embedding_metadata.json')

EMBEDDER_MODEL_NAME = 'Qwen/Qwen3-Embedding-0.6B'
MAX_SEQ_LENGTH = 1024   # matches the reranker's max_length — embedder truncates internally at this point

os.makedirs(os.path.dirname(RAW_CSV), exist_ok=True)
print('Config set.')


Config set.


## 3 · Download Raw Dataset

In [3]:
import opendatasets as od

if not os.path.exists(RAW_CSV):
    od.download(KAGGLE_DATASET, data_dir=os.path.dirname(RAW_CSV))
    for root, _, files in os.walk(os.path.dirname(RAW_CSV)):
        for f in files:
            if f.lower().endswith('.csv'):
                print('Found:', os.path.join(root, f))
else:
    print('Raw CSV already present, skipping download.')


Raw CSV already present, skipping download.


## 4 · Clean & Preprocess

Same robust encoding handling established during local debugging: plain UTF-8 first, widen
`charset_normalizer`'s scan if that fails, lossy fallback (with a data-loss count printed) only
as a last resort. Light-touch cleaning only — NFKC normalization + scraping-artifact removal,
no stemming/stopword removal/diacritic stripping.

In [4]:
def load_csv_robust(path: str) -> pd.DataFrame:
    try:
        df = pd.read_csv(path, encoding='utf-8')
        print('Loaded as plain UTF-8.')
        return df
    except UnicodeDecodeError as e:
        print(f'utf-8 failed at byte {e.start}: {e.reason} — widening charset_normalizer scan...')
    except pd.errors.ParserError as e:
        print(f'utf-8 decoded fine but CSV parsing failed ({e}).')

    detected = detect_encoding(path, steps=40, chunk_size=4096).best()
    if detected is not None and detected.encoding:
        print(f'Detected encoding: {detected.encoding}')
        return pd.read_csv(path, encoding=detected.encoding)

    print('No confident encoding found — falling back to lossy utf-8.')
    return pd.read_csv(path, encoding='utf-8', encoding_errors='replace')


def clean_text(s: str) -> str:
    if not isinstance(s, str):
        return ''
    s = ftfy.fix_text(s)
    s = unicodedata.normalize('NFKC', s)
    s = re.sub(r'&[a-zA-Z]+;|&#\d+;', ' ', s)
    s = re.sub(r'\[\d+\]', ' ', s)
    s = re.sub(r'[\u200b-\u200f\ufeff]', '', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s


df = load_csv_robust(RAW_CSV)
for col in ('titles', 'questions', 'answers'):
    df[col] = df[col].fillna('').apply(clean_text)

n_bad = int(df['titles'].str.count('\ufffd').sum() + df['answers'].str.count('\ufffd').sum())
print(f'Rows: {len(df):,}  |  Replacement-char count (data loss indicator): {n_bad}')

# Stable id + source attribution — the metadata that travels alongside the embeddings
df = df.reset_index(drop=True)
df.insert(0, 'fatwa_id', df.index)
df.insert(1, 'source', SOURCE_LABEL)

df.rename(columns={'titles': 'title', 'questions': 'question', 'answers': 'answer'}, inplace=True)
df.head(3)


Loaded as plain UTF-8.
Rows: 19,727  |  Replacement-char count (data loss indicator): 0


,fatwa_id,source,question,answer,title,links,categories
0,0,Ibn Baz,يقول تعالى: لِيَبْلُوَكُمْ أَيُّكُمْ أَحْسَنُ ...,هذا يدل على أنَّ الحُسن أهم، إحسان العمل أهم م...,معنى قوله تعالى: {ليبلوكم أيُّكم أحسنُ عملاً},https://binbaz.org.sa/fatwas/2746/معنى-قوله-تع...,['التفسير']
1,1,Ibn Baz,مجموعة أسئلة يا فضيلة الشيخ تدور حول الأغاني، ...,أما الأغاني: فقد ابتُلي بها الناسُ اليوم، وكثر...,ما النصيحة لمن يستمع الأغاني؟,https://binbaz.org.sa/fatwas/2747/ما-النصيحة-ل...,['الشعر والأغاني']
2,2,Ibn Baz,أولًا: أرجو منك أن تدعو لي بالفقه في الدين.ثان...,أولًا: رزقنا الله وإياك الفقه في الدين، وأصلح ...,هل الأفضل صلاة النافلة خفيةً عن الناس؟,https://binbaz.org.sa/fatwas/2748/هل-الافضل-صل...,['صلاة التطوع']


## 5 · Build `embed_text` + Truncation Metadata

`embed_text` = `title + answer` — the unit that actually gets embedded and, later, reranked.
The embedder truncates internally at `MAX_SEQ_LENGTH` (set on the model in §6), so this step
doesn't cut any text — it only **measures and records** which rows exceed the budget, so you
have visibility into how much of the tail is affected, without silently losing anything from
`binbaz_clean.csv` itself (the full, untruncated `answer` column is always preserved).

In [5]:
tokenizer = SentenceTransformer(EMBEDDER_MODEL_NAME, device='cpu').tokenizer  # cheap: CPU load just for the tokenizer

df['embed_text'] = df['title'] + ' — ' + df['answer']

token_counts = [len(ids) for ids in tokenizer(df['embed_text'].tolist(), add_special_tokens=False)['input_ids']]
df['embed_token_count'] = token_counts
df['truncated'] = df['embed_token_count'] > MAX_SEQ_LENGTH

n_truncated = int(df['truncated'].sum())
print(f'{n_truncated:,} / {len(df):,} rows ({n_truncated/len(df)*100:.1f}%) exceed {MAX_SEQ_LENGTH} tokens '
      'and will be truncated for embedding/reranking — their full answer stays intact in `answer`.')

del tokenizer
gc.collect()


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.19G [00:00<?, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

864 / 19,727 rows (4.4%) exceed 1024 tokens and will be truncated for embedding/reranking — their full answer stays intact in `answer`.


6766

## 6 · Multi-GPU Embedding Generation

Uses `SentenceTransformer`'s multi-process pool to split encoding across both T4s. `max_seq_length`
set on the model itself so truncation happens automatically inside `.encode()` — no manual
text-truncation step needed.

In [6]:
embedder = SentenceTransformer(EMBEDDER_MODEL_NAME, device='cuda')
embedder.max_seq_length = MAX_SEQ_LENGTH

texts = df['embed_text'].tolist()
print(f'Encoding {len(texts):,} texts with {EMBEDDER_MODEL_NAME} (max_seq_length={MAX_SEQ_LENGTH})...')

t0 = time.time()
if torch.cuda.device_count() >= 2:
    pool = embedder.start_multi_process_pool()
    embeddings_np = embedder.encode_multi_process(texts, pool, batch_size=32, show_progress_bar=True)
    embedder.stop_multi_process_pool(pool)
else:
    embeddings_np = embedder.encode(texts, batch_size=32, show_progress_bar=True)
elapsed = time.time() - t0

embeddings = torch.tensor(embeddings_np)
print(f'Done in {elapsed/60:.1f} min. Shape: {tuple(embeddings.shape)}')


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Encoding 19,727 texts with Qwen/Qwen3-Embedding-0.6B (max_seq_length=1024)...


/tmp/ipykernel_58/1107570714.py:10: DeprecationWarning: The `encode_multi_process` method has been deprecated, and its functionality has been integrated into `encode`. You can now call `encode` with the same parameters to achieve multi-process encoding.
  embeddings_np = embedder.encode_multi_process(texts, pool, batch_size=32, show_progress_bar=True)


Chunks:   0%|          | 0/20 [00:00<?, ?it/s]

Done in 33.9 min. Shape: (19727, 1024)


## 7 · Save & Verify Outputs

In [7]:
assert embeddings.shape[0] == len(df), 'Embedding count and row count must match — check for silent drops.'

df.to_csv(CLEAN_CSV, index=False)
torch.save(embeddings, EMBEDDINGS_PT)

metadata = {
    'model': EMBEDDER_MODEL_NAME,
    'embed_dim': int(embeddings.shape[1]),
    'max_seq_length': MAX_SEQ_LENGTH,
    'n_rows': int(embeddings.shape[0]),
    'n_truncated': n_truncated,
    'source': SOURCE_LABEL,
    'generated_at_utc': datetime.now(timezone.utc).isoformat(),
}
with open(METADATA_JSON, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print(f'Saved:\n  {CLEAN_CSV}\n  {EMBEDDINGS_PT}\n  {METADATA_JSON}')
print('\nDownload all three from the Output tab on the right once this run finishes.')
print(json.dumps(metadata, ensure_ascii=False, indent=2))


Saved:
  /kaggle/working/binbaz_clean.csv
  /kaggle/working/qwen3_06b_title_answer_embeddings.pt
  /kaggle/working/embedding_metadata.json

Download all three from the Output tab on the right once this run finishes.
{
  "model": "Qwen/Qwen3-Embedding-0.6B",
  "embed_dim": 1024,
  "max_seq_length": 1024,
  "n_rows": 19727,
  "n_truncated": 864,
  "source": "Ibn Baz",
  "generated_at_utc": "2026-09-08T13:26:56.191210+00:00"
}


## Published artifacts

The three output files are published on the Hugging Face dataset [`ZenoAbderrahim/binbaz_fatwas_rag`](https://huggingface.co/datasets/ZenoAbderrahim/binbaz_fatwas_rag). `app.py` downloads them automatically into `data/` on first run, so you only need this notebook to rebuild them.